# Feature Engineering and Model Preparation

This notebook prepares the cleaned BoardGameGeek dataset for machine-learning modelling.

The goal of this phase is to transform the cleaned dataset into modelling-ready inputs.

This notebook will:

* load and verify the cleaned dataset;
* define the predicion target;
* select the initial feature columns;
* exclude identifier, ranking, and post-release popularity columns;
* prepare numerical features;
* prepare text-based features from `Mechanics` and `Domains`;
* create the feature matrix `X` and target variable `y`:
* split the data into training and testing sets;
* save perpared modelling datasets;
* document the feature engineering decisions.


## Load and Verify Cleaned Dataset for Modelling

The cleaned BoardGameGeek dataset is loaded from the `data/processed` folder.

Before feature engineering begins, the dataset is checked to confirm that the correct proccessed file is being used.

This step verifies:

* the cleaned file path;
* the dataset shape;
* the expected column names;
* dupilacte rows;
* missing values in the prediction target.


In [477]:
# Import libraries used throughout this current notebook
import numpy as np
import pandas as pd

# Import Path for creating file paths that work
# reliebly across different OS-s
from pathlib import Path


# Find the folder from where which this notebook is running
current_folder = Path.cwd()

# The notebook is inside the "notebooks" folder,
# so it parent folder is the main project folder
project_root = current_folder.parent

# Create the complete path to the cleaned dataset
cleaned_data_path = (
    project_root
    / "data"
    / "processed"
    / "bgg_cleaned.csv"
)

print("Current folder:", current_folder)
print("Project root:", project_root)
print("Cleaned dataset path:", cleaned_data_path)
print("File exists:", cleaned_data_path.exists())

Current folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor
Cleaned dataset path: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed\bgg_cleaned.csv
File exists: True


In [478]:
# Load cleaned dataset into DataFrame for modelling preparation
df_model = pd.read_csv(cleaned_data_path)

print("Dataset loaded succeesfully.")
print("Dataset shape:", df_model.shape)


Dataset loaded succeesfully.
Dataset shape: (20342, 14)


In [479]:
# Display first 5 rows of the modelling 
# dataset (confirmation that we are good to go)
df_model.head() 

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1.0,4.0,120.0,14.0,42055,8.79,1,3.86,68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2.0,4.0,60.0,13.0,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2.0,4.0,120.0,14.0,19217,8.66,3,3.91,28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1.0,5.0,120.0,12.0,64864,8.43,4,3.24,87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3.0,6.0,480.0,14.0,13468,8.70,5,4.22,16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [480]:
# Display random 5 rows of modelling 
# dataset (as I want to be extra sure)
df_model.sample(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
7976,262498.0,Puzzle Dungeon,2019.0,1.0,1.0,20.0,13.0,77,7.72,7978,1.80,191.0,"Hand Management, Modular Board, Set Collection...",Unknown
8163,319114.0,Krazy Pix,2020.0,3.0,8.0,45.0,10.0,63,7.40,8165,1.00,196.0,Unknown,Unknown
4796,330.0,Block Mania,1987.0,2.0,2.0,180.0,14.0,427,6.48,4798,2.75,866.0,"Action Points, Dice Rolling",Thematic Games
13385,2160.0,Robots!,1980.0,2.0,4.0,120.0,10.0,61,6.17,13387,2.63,211.0,"Dice Rolling, Hexagon Grid",Wargames
14611,3540.0,Tic Tac Chec,1995.0,2.0,2.0,10.0,NaN,68,6.00,14614,2.17,202.0,"Grid Movement, Pattern Building",Abstract Games


In [481]:
# Deifne the expected column names
expected_columns = [
    "ID",
    "Name",
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Users Rated",
    "Rating Average",
    "BGG Rank",
    "Complexity Average",
    "Owned Users",
    "Mechanics",
    "Domains"
]


# Verify that the correct cleaned dataset has been loaded
print("Correct number of rows:", df_model.shape[0] == 20342)
print("Correct number of columns:", df_model.shape[1] == 14)

print(
    "Column names match:",
    df_model.columns.tolist() == expected_columns
)

print("Fully duplicated rows:", df_model.duplicated().sum())

print(
    "Missing Rating Average values:",
    df_model["Rating Average"].isnull().sum()
) 

Correct number of rows: True
Correct number of columns: True
Column names match: True
Fully duplicated rows: 0
Missing Rating Average values: 0


### Cleaned Dataset Loading Resutls

The cleaned dataset was loaded successfully from `data/processed/bgg_cleaned.csv`.

The dataset contain 20,342 rows and 14 columns, matching the expected processd dataset created during the data cleaning phase.

The expected column names are present, there are no fully duplicated rows, and the prediction target `Rating Average` contains no missing values.

This confirms that the feature engineering notebook is starting from the correct cleaned dataset.


## Define Target and Feature Columns

The prediction target an possible feature columns are defined before modelling preparation begins.

The target is the value the machine-learning model will try to predict.

The feature columns are the information the model may use to make that prediction.

For this proejct, the prediction target is `Rating Average`.

The initial feature plan focuses on board game design information such as publicaton year, player counts, play time, recommended age, complexity, mehanics, and domains.


In [482]:
# Define prediction target
target_column = "Rating Average"

print("Target column:", target_column)
print("Target data type:", df_model[target_column].dtype)
print("Missing target values:", df_model[target_column].isnull().sum())

Target column: Rating Average
Target data type: float64
Missing target values: 0


In [483]:
# Define initial numerical feature candidates
numeric_feature_candidates = [
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Complexity Average"
]

# Define initial text-based feature candidates
text_feature_candidates = [
    "Mechanics",
    "Domains"
]

# Define columns that need to be reviewed for excluson
columns_to_review_for_exclusion = [
    "ID",
    "Name",
    "BGG Rank",
    "Users Rated",
    "Owned Users"
]

print("Numerical feature candidates:", numeric_feature_candidates)
print("Text feature candidates:", text_feature_candidates)
print("Columns to review for exclusion:", columns_to_review_for_exclusion)

Numerical feature candidates: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average']
Text feature candidates: ['Mechanics', 'Domains']
Columns to review for exclusion: ['ID', 'Name', 'BGG Rank', 'Users Rated', 'Owned Users']


In [484]:
# Create summary table of the target and feature groups
feature_definition_summary = pd.DataFrame(
    [
        {
            "Column": target_column,
            "Role": "Target",
            "Reason": "Value the model will predict"
        },
        *[
            {
                "Column": column,
                "Role": "Numerical feature candidate",
                "Reason": "Board game design or description information"
            }
            for column in numeric_feature_candidates
        ],
        *[
            {
                "Column": column,
                "Role": "Text feature candidate",
                "Reason": "Multi-value board game category information"
            }
            for column in text_feature_candidates
        ],
        *[
            {
                "Column": column,
                "Role": "Review for exclusion",
                "Reason": "Identifier, name, ranjing, or post-release popularity information"
            }
            for column in columns_to_review_for_exclusion
        ]
    ]
)

feature_definition_summary

,Column,Role,Reason
0,Rating Average,Target,Value the model will predict
1,Year Published,Numerical feature candidate,Board game design or description information
2,Min Players,Numerical feature candidate,Board game design or description information
3,Max Players,Numerical feature candidate,Board game design or description information
4,Play Time,Numerical feature candidate,Board game design or description information
5,Min Age,Numerical feature candidate,Board game design or description information
6,Complexity Average,Numerical feature candidate,Board game design or description information
7,Mechanics,Text feature candidate,Multi-value board game category information
8,Domains,Text feature candidate,Multi-value board game category information
9,ID,Review for exclusion,"Identifier, name, ranjing, or post-release pop..."


In [485]:
# Combine all defined columns into the one list
defined_columns = (
    [target_column]
    + numeric_feature_candidates
    + text_feature_candidates
    + columns_to_review_for_exclusion
)

# Check that for any columns that were defined but are not present in the dataset
missing_defined_columns = [
    column
    for column in defined_columns
    if column not in df_model.columns
]

# Check that all dataset columns have been assigned to the one of the groups
unassigned_columns = [
    column
    for column in df_model.columns
    if column not in defined_columns
]

print("Missing defined columns:", missing_defined_columns)
print("Unassigned dataset columns:", unassigned_columns)
print("Total defined columns:", len(defined_columns))
print("Total dataset columns:", len(df_model.columns))

Missing defined columns: []
Unassigned dataset columns: []
Total defined columns: 14
Total dataset columns: 14


### Target and Feature Column Definition Result

The prediction target was defind as `Rating Average`.

The initial numerical feature candidates are:

* `Year Published`
* `Min Players`
* `Max Players`
* `Play Tine`
* `Min Age`
* `Complexity Average`

The initial text-based feature candidates are:

* `Mechanics`
* `Domains`

The following columns were idetified for exclusion review:

* `ID`
* `Name`
* `BGG Rank`
* `Users Rated`
* `Owned Users`

All 14 columns in the cleaned dataset have been assigned to the target, feature candidate, or exclusion-review group.

Not data was changed during this step.


## Exclude Leakage and Identifier Columns

Some columns should not be used as model features.

Idebtifier columns, ranking columns, and post-release popularity columns may either provide no usefoul learning signal or give the model information that would not be available before a game receives public ratings.

This step separates useable feature columns from columns that should be excluded from the initial modelling dataset.


In [486]:
# Define columns that will be excluded from the initial model features
excluded_columns = {
    "ID": "Identifier column; does not descirbe board game design.",
    "Name": "Game title; may cause the model to memorise specific games rather than learn general patterns.",
    "BGG Rank": "Ranking infromation based on BoardGameGeek performance; likely target leakage.",
    "Users Rated": "Post-release popularity information; not suitable for initial pre-release-style prediction.",
    "Owned Users": "Post-release ownership/popularity information; not suitable for initial pre-release-style prediction."
}

excluded_columns

{'ID': 'Identifier column; does not descirbe board game design.',
 'Name': 'Game title; may cause the model to memorise specific games rather than learn general patterns.',
 'BGG Rank': 'Ranking infromation based on BoardGameGeek performance; likely target leakage.',
 'Users Rated': 'Post-release popularity information; not suitable for initial pre-release-style prediction.',
 'Owned Users': 'Post-release ownership/popularity information; not suitable for initial pre-release-style prediction.'}

In [487]:
# Create a readable table explaining why each 
# of the column is excluded
exclusion_summary = pd.DataFrame(
    [
        {
            "Column": column,
            "Reason for Exculsion": reason
        }
        for column, reason in excluded_columns.items()
    ]
)

exclusion_summary

,Column,Reason for Exculsion
0,ID,Identifier column; does not descirbe board gam...
1,Name,Game title; may cause the model to memorise sp...
2,BGG Rank,Ranking infromation based on BoardGameGeek per...
3,Users Rated,Post-release popularity information; not suita...
4,Owned Users,Post-release ownership/popularity information;...


In [488]:
# Define selectd feature columns for the initial model
selected_numeric_features = numeric_feature_candidates.copy()
selected_text_features = text_feature_candidates.copy()

selected_feature_columns = (
    selected_numeric_features
    + selected_text_features
)

print("Selected numerical features:", selected_numeric_features)
print("Selected text features:", selected_text_features)
print("All of selected feature columns:", selected_feature_columns)

Selected numerical features: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average']
Selected text features: ['Mechanics', 'Domains']
All of selected feature columns: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average', 'Mechanics', 'Domains']


In [489]:
# Check that excluded columns are not present in 
# the selected feature columns -fail safes
excluded_columns_in_features = [
    column
    for column in excluded_columns
    if column in selected_feature_columns
]

# Check that the target column is not accidentally included as a feature
target_in_features = target_column in selected_feature_columns

print("Exluded columns accidentally included as features:", excluded_columns_in_features)
print("Target column accidentally included as feature:", target_in_features)
print("Number of selected feature columns:", len(selected_feature_columns))

Exluded columns accidentally included as features: []
Target column accidentally included as feature: False
Number of selected feature columns: 8


In [490]:
# Create modelling column summary
modelling_column_summary = pd.DataFrame(
    [
        {
            "Column": target_column,
            "Modelling Role": "Target"
        },
        *[
            {
                "Column": column,
                "Modelling Role": "Selected numerical feature"
            }
            for column in selected_numeric_features
        ],
        *[
            {
                "Column": column,
                "Modelling Role": "Selected text feature"
            }
            for column in selected_text_features
        ],
        *[
            {
                "Column": column,
                "Modelling Role": "Excluded"
            }
            for column in excluded_columns
        ]
    ]
)

modelling_column_summary

,Column,Modelling Role
0,Rating Average,Target
1,Year Published,Selected numerical feature
2,Min Players,Selected numerical feature
3,Max Players,Selected numerical feature
4,Play Time,Selected numerical feature
5,Min Age,Selected numerical feature
6,Complexity Average,Selected numerical feature
7,Mechanics,Selected text feature
8,Domains,Selected text feature
9,ID,Excluded


### Leakage and Identifier Column Exclusion Result

The initial model feature set excludes `ID`, `Name`, `BGG Rank`, `Users Rated`, and `Owned Users`.

`ID` is excluded because it is only an identifier and does not help us as it does not describe board game design.

`Name` is excluded because game titles may encourage memorisation (relation to brand, movie, TV show, book) rather than general learning.

`BGG Rank` is excluded because it is ranking information closely related to rating performance and may introduce target leakage.

`Users Rated` and `Owned Users` are excluded because they describe post-release popularity. These values would not be suitable for an initial model focused on predicting ratings from board game design information.

The selected initial features are six numericla columns and two text-based columns.

Not a single excluded columns was accidentally included in the selected feature list, and the target column was not included as a feature.


## Prepare Numerical Features

The numerical feature columns are prepered for later modelling.

This step creates a separate numerical feature working table, adds missing-value indicator columns, and creates log-transformd versions of heavily skewed features.

Missing numerical values are not filled during this step. They will be handeld later after the train/test split so that imputation values are learned from the training data only.


In [491]:
# Create seprate working table for numerical model features
numeric_model_data = df_model[selected_numeric_features].copy()

print("Numerical model data shape:", numeric_model_data.shape)

# 5 values
numeric_model_data.head()
# 7 values
numeric_model_data.head(7)


Numerical model data shape: (20342, 6)


,Year Published,Min Players,Max Players,Play Time,Min Age,Complexity Average
0,2017.0,1.0,4.0,120.0,14.0,3.86
1,2015.0,2.0,4.0,60.0,13.0,2.84
2,2018.0,2.0,4.0,120.0,14.0,3.91
3,2016.0,1.0,5.0,120.0,12.0,3.24
4,2017.0,3.0,6.0,480.0,14.0,4.22
5,2020.0,1.0,4.0,120.0,14.0,3.55
6,2015.0,2.0,4.0,120.0,14.0,4.41


In [492]:
# Review missing values in selectzed numerical features
numeric_model_missing_summary = numeric_model_data.isnull().sum().to_frame(
    name="Missing Values"
)

numeric_model_missing_summary["Missing Percentage"] = (
    numeric_model_missing_summary["Missing Values"]
    / len(numeric_model_data)
    * 100
).round(2)

numeric_model_missing_summary

,Missing Values,Missing Percentage
Year Published,185,0.91
Min Players,46,0.23
Max Players,161,0.79
Play Time,556,2.73
Min Age,1250,6.14
Complexity Average,426,2.09


In [493]:
# Identify numerical columns that contain missing values
numeric_columns_with_missing_values = [
    column
    for column in selected_numeric_features
    if numeric_model_data[column].isnull().sum() > 0
]

# Create missing-value indicator columns
# These columns store 1 when original value was missing,
# or 0 when original value was present
missing_indicator_features = []

for column in numeric_columns_with_missing_values:
    indicator_column = f"{column} Missing"

    numeric_model_data[indicator_column] = (
        numeric_model_data[column]
        .isnull()
        .astype(int)
    )

    missing_indicator_features.append(indicator_column)

print("NUmeric columns with missing values:", numeric_columns_with_missing_values)
print("Missing indicator features:", missing_indicator_features)

NUmeric columns with missing values: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average']
Missing indicator features: ['Year Published Missing', 'Min Players Missing', 'Max Players Missing', 'Play Time Missing', 'Min Age Missing', 'Complexity Average Missing']


In [494]:
# Create log-transformed versions of heavily skewed numerical features
# this will br useful because 
# 1. very large Play Time & Max player values will become less extreme after log transformation
# 2. np.log1p() safely handling values like 0 if they ever appear
numeric_model_data["Max Players Log"] = np.log1p(
    numeric_model_data["Max Players"]
)

numeric_model_data["Play Time Log"] = np.log1p(
    numeric_model_data["Play Time"]
)

numeric_model_data[
    [
        "Max Players",
        "Max Players Log",
        "Play Time",
        "Play Time Log"
    ]
].head()

,Max Players,Max Players Log,Play Time,Play Time Log
0,4.0,1.609438,120.0,4.795791
1,4.0,1.609438,60.0,4.110874
2,4.0,1.609438,120.0,4.795791
3,5.0,1.791759,120.0,4.795791
4,6.0,1.945910,480.0,6.175867


In [495]:
# Define numerical features that will be used for modelling
numeric_features_for_model = [
    "Year Published",
    "Min Players",
    "Min Age",
    "Complexity Average",
    "Max Players Log",
    "Play Time Log"
] + missing_indicator_features

print("Number of numerical features for model:", len(numeric_features_for_model))

numeric_features_for_model

Number of numerical features for model: 12


['Year Published',
 'Min Players',
 'Min Age',
 'Complexity Average',
 'Max Players Log',
 'Play Time Log',
 'Year Published Missing',
 'Min Players Missing',
 'Max Players Missing',
 'Play Time Missing',
 'Min Age Missing',
 'Complexity Average Missing']

In [496]:
# Check missing values in the final numerical feature set
numeric_features_for_model_missing_summary = (
    numeric_model_data[numeric_features_for_model]
    .isnull()
    .sum()
    .to_frame(name="Missing Values")
)

numeric_features_for_model_missing_summary

,Missing Values
Year Published,185
Min Players,46
Min Age,1250
Complexity Average,426
Max Players Log,161
Play Time Log,556
Year Published Missing,0
Min Players Missing,0
Max Players Missing,0
Play Time Missing,0


### Numerical Feature Preparation Result

A separate numerical feature working table was created from the selected numerical feature columns.

Missing-value indicator columns were added for numerical features that contain missing values. These indicators record wheather the original value was missing.

Log-transformed versions of `Max Players` and `Play Time` were created because these columns were heavily right-skewed during exploratory data analysis.

The initial numerical feature set for modelling uses the log-transformed versions of `Max Players` anf `Play Time` instead of the raw values.

Missing numerical values were not filled during this step. They will be handled later after the train/test spilt so that imputaton values are learned from the training data only.


## Prepare Mechanics and Domains Features

Plan is following:
Task prepares the two text-based columns:
The `Mechanics` and `Domains` columns contain text-based, mutil-value information.

These columns are not directly usable by most machine-learning models because they contain text, and sometimes also several values inside one cell.
Machine-learning models need numerical inputs, so these columns are transformed into binary indicator features.

SInce model cannot use that raw text directly, so we will create simple 0/1 indicator column
Value of `1` means that a game contains a specific mechanic or domain. Value of `0` means that it does not.

For the initial model preparation, the most common mechanics are selected, while all available non-unknown domains are prepared as domain indicator features.

In [497]:
# Create a separate working table for text-based model features
text_model_data = df_model[selected_text_features].copy()

print("Text model data shape:", text_model_data.shape)

# Display the first five rows of the text feature working table
text_model_data.head()

Text model data shape: (20342, 2)


,Mechanics,Domains
0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [498]:
# Count Unknown values in the selected text features
text_unknown_summary = pd.DataFrame(
    {
        "Unknown Count": [
            (text_model_data["Mechanics"] == "Unknown").sum(),
            (text_model_data["Domains"] == "Unknown").sum()
        ],
        "Unknown Percentage": [
            (
                (text_model_data["Mechanics"] == "Unknown").sum()
                / len(text_model_data)
                * 100
            ).round(2),
            (
                (text_model_data["Domains"] == "Unknown").sum()
                / len(text_model_data)
                * 100
            ).round(2)
        ]
    },
    index=["Mechanics", "Domains"]
)

text_unknown_summary

,Unknown Count,Unknown Percentage
Mechanics,1597,7.85
Domains,10158,49.94


In [499]:
# Split Mechanics into individual mechanic values
mechanics_values = (
    text_model_data["Mechanics"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Select top 15 most common mechanics but exclude Unknown
# Starting with most common as there might be a lot and 
# it is good to keep it simpel 
top_mechanics_for_model = (
    mechanics_values[mechanics_values != "Unknown"]
    .value_counts()
    .head(15)
    .index
    .tolist()
)

top_mechanics_for_model

['Dice Rolling',
 'Hand Management',
 'Set Collection',
 'Variable Player Powers',
 'Hexagon Grid',
 'Simulation',
 'Card Drafting',
 'Tile Placement',
 'Modular Board',
 'Area Majority / Influence',
 'Grid Movement',
 'Cooperative Game',
 'Roll / Spin and Move',
 'Area Movement',
 'Simultaneous Action Selection']

In [500]:
# Split Domains into individual domain values
domain_values = (
    text_model_data["Domains"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Select all available domains butr exclud Unknown
domains_for_model = (
    domain_values[domain_values != "Unknown"]
    .value_counts()
    .index
    .tolist()
)

domains_for_model

['Wargames',
 'Strategy Games',
 'Family Games',
 'Thematic Games',
 'Abstract Games',
 "Children's Games",
 'Party Games',
 'Customizable Games']

In [501]:
# Create binary indicator columns for selected mechanics
mechanic_indicator_features = []

for mechanic in top_mechanics_for_model:
    indicator_column = f"Mechanic: {mechanic}"

    text_model_data[indicator_column] = (
        text_model_data["Mechanics"]
        .fillna("")
        .str.contains(mechanic, regex=False)
        .astype(int)
    )

    mechanic_indicator_features.append(indicator_column)

print("Number of mechanic indicator features:", len(mechanic_indicator_features))

mechanic_indicator_features

Number of mechanic indicator features: 15


['Mechanic: Dice Rolling',
 'Mechanic: Hand Management',
 'Mechanic: Set Collection',
 'Mechanic: Variable Player Powers',
 'Mechanic: Hexagon Grid',
 'Mechanic: Simulation',
 'Mechanic: Card Drafting',
 'Mechanic: Tile Placement',
 'Mechanic: Modular Board',
 'Mechanic: Area Majority / Influence',
 'Mechanic: Grid Movement',
 'Mechanic: Cooperative Game',
 'Mechanic: Roll / Spin and Move',
 'Mechanic: Area Movement',
 'Mechanic: Simultaneous Action Selection']

In [502]:
# Create binary indicator columns for selected domains
domain_indicator_features = []

for domain in domains_for_model:
    indicator_column = f"Domain: {domain}"

    text_model_data[indicator_column] = (
        text_model_data["Domains"]
        .fillna("")
        .str.contains(domain, regex=False)
        .astype(int)
    )

    domain_indicator_features.append(indicator_column)

print("Number of domain indicator features:", len(domain_indicator_features))

domain_indicator_features

Number of domain indicator features: 8


['Domain: Wargames',
 'Domain: Strategy Games',
 'Domain: Family Games',
 'Domain: Thematic Games',
 'Domain: Abstract Games',
 "Domain: Children's Games",
 'Domain: Party Games',
 'Domain: Customizable Games']

In [503]:
# Preview the created text indicator features
# num of mechanics (15) + domains (8)
text_features_for_model = (
    mechanic_indicator_features
    + domain_indicator_features
)

print("Total number of text features for model:", len(text_features_for_model))

text_model_data[text_features_for_model].head()

Total number of text features for model: 23


,Mechanic: Dice Rolling,Mechanic: Hand Management,Mechanic: Set Collection,Mechanic: Variable Player Powers,Mechanic: Hexagon Grid,Mechanic: Simulation,Mechanic: Card Drafting,Mechanic: Tile Placement,Mechanic: Modular Board,Mechanic: Area Majority / Influence,...,Mechanic: Area Movement,Mechanic: Simultaneous Action Selection,Domain: Wargames,Domain: Strategy Games,Domain: Family Games,Domain: Thematic Games,Domain: Abstract Games,Domain: Children's Games,Domain: Party Games,Domain: Customizable Games
0,0,1,0,1,1,0,0,0,1,0,...,0,1,0,1,0,1,0,0,0,0
1,0,1,1,1,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0,1,1,1,1,0,1,1,0,0,...,0,0,0,1,0,0,0,0,0,0
4,1,0,0,1,1,0,0,0,1,1,...,0,0,0,1,0,1,0,0,0,0


In [504]:
# Check missing values in the final text feature set
text_features_for_model_missing_summary = (
    text_model_data[text_features_for_model]
    .isnull()
    .sum()
    .to_frame(name="Missing Values")
)

text_features_for_model_missing_summary

,Missing Values
Mechanic: Dice Rolling,0
Mechanic: Hand Management,0
Mechanic: Set Collection,0
Mechanic: Variable Player Powers,0
Mechanic: Hexagon Grid,0
Mechanic: Simulation,0
Mechanic: Card Drafting,0
Mechanic: Tile Placement,0
Mechanic: Modular Board,0
Mechanic: Area Majority / Influence,0


### Mechanics and Domains Feature Preparation Result

A separate text feature working table was created from the `Mechanics` and `Domains` columns.

The `Mechanics` column was transformed into binary indicator features for the 15 most common mechanics.

The `Domains` column was transformed into binary indicator features for all available non-unknown domains, and its value was 8.

Each created feature uses `1` when a game contains that mechanic or domain and `0` when they do not.

The original text columns were not removed this time around from the raw modelling working table, but created indicator columns are the derived features intendend for modelling.

The created text indicator features contain no missing values.

## Create Feature Matrix and Target Variable

The prepared numerical and text-derived features are combined into a single feature matrix.

In machine learning, the feature matrix is commonly named `x`, while the prediction target is commonlky named `y`.

For this project:

- `x` contains the prepared board game features used for prediction.
- `y` contains the `Rating Average` values that the model will try to predict.

The excluded columns are not included in `X`.

In [505]:
# Combine numerical and text-derived features into one final feature lsit
features_for_model = (
    numeric_features_for_model
    + text_features_for_model
)

print("Number of numerical features:", len(numeric_features_for_model))
print("Number of text-derived features:", len(text_features_for_model))
print("Total number of featurs:", len(features_for_model))

features_for_model

Number of numerical features: 12
Number of text-derived features: 23
Total number of featurs: 35


['Year Published',
 'Min Players',
 'Min Age',
 'Complexity Average',
 'Max Players Log',
 'Play Time Log',
 'Year Published Missing',
 'Min Players Missing',
 'Max Players Missing',
 'Play Time Missing',
 'Min Age Missing',
 'Complexity Average Missing',
 'Mechanic: Dice Rolling',
 'Mechanic: Hand Management',
 'Mechanic: Set Collection',
 'Mechanic: Variable Player Powers',
 'Mechanic: Hexagon Grid',
 'Mechanic: Simulation',
 'Mechanic: Card Drafting',
 'Mechanic: Tile Placement',
 'Mechanic: Modular Board',
 'Mechanic: Area Majority / Influence',
 'Mechanic: Grid Movement',
 'Mechanic: Cooperative Game',
 'Mechanic: Roll / Spin and Move',
 'Mechanic: Area Movement',
 'Mechanic: Simultaneous Action Selection',
 'Domain: Wargames',
 'Domain: Strategy Games',
 'Domain: Family Games',
 'Domain: Thematic Games',
 'Domain: Abstract Games',
 "Domain: Children's Games",
 'Domain: Party Games',
 'Domain: Customizable Games']

In [506]:
# Combine prepared numerical features an text-derived features
prepared_feture_data = pd.concat(
    [
        numeric_model_data[numeric_features_for_model],
        text_model_data[text_features_for_model]
    ],
    axis=1
)

print("Prepared feature data shape:", prepared_feture_data.shape)

prepared_feture_data.head()

Prepared feature data shape: (20342, 35)


,Year Published,Min Players,Min Age,Complexity Average,Max Players Log,Play Time Log,Year Published Missing,Min Players Missing,Max Players Missing,Play Time Missing,...,Mechanic: Area Movement,Mechanic: Simultaneous Action Selection,Domain: Wargames,Domain: Strategy Games,Domain: Family Games,Domain: Thematic Games,Domain: Abstract Games,Domain: Children's Games,Domain: Party Games,Domain: Customizable Games
0,2017.0,1.0,14.0,3.86,1.609438,4.795791,0,0,0,0,...,0,1,0,1,0,1,0,0,0,0
1,2015.0,2.0,13.0,2.84,1.609438,4.110874,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
2,2018.0,2.0,14.0,3.91,1.609438,4.795791,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,2016.0,1.0,12.0,3.24,1.791759,4.795791,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,2017.0,3.0,14.0,4.22,1.945910,6.175867,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0


In [507]:
# Create feature matrix X and the target variable y
X = prepared_feture_data.copy()
y = df_model[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20342, 35)
y shape: (20342,)


In [508]:
# Check if EXCLUDED columns are not present in X
excluded_columns_in_X = [
    column
    for column in excluded_columns
    if column in X.columns
]

# Check if the TAREGT column is not present in X
target_column_in_X = target_column in X.columns

print("Excluded columns found in X:", excluded_columns_in_X)
print("Target column found in X:", target_column_in_X)

Excluded columns found in X: []
Target column found in X: False


In [509]:
# Chexk missing values in X an y
print("Total missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())

Total missing values in X: 2624
Missing values in y: 0


### Feature Matrix and Target Variable Result

The numerical and text-derived features were combined into one prepared feature table.

The final feature matrix `X` contains 20,342 rows and 35 feature columns.

The target variable `y` contains 20,342 `Rating Average` values

The exculed columns were not included in `X`, and the target column was not accidentally included as a feature.

The target variable contains not missing values.

Some missing values remain in `X`, but that is because numerical missing values have not been imputed yet. These values will be handled after the train/test split so that imputation values are learned from the training data only.